In [1]:
import os
os.chdir(r"C:\Users\Lenovo\Desktop\KYC_Project")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Lenovo\Desktop\KYC_Project


In [2]:
!pip install mediapipe
!pip install onnxruntime

  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/20.1 MB 825.8 kB/s eta 0:00:25
   ---------------------------------------- 0.1/20.1 MB 1.2 MB/s eta 0:00:17
    --------------------------------------- 0.3/20.1 MB 1.6 MB/s eta 0:00:13
    --------------------------------------- 0.5/20.1 MB 2.6 MB/s eta 0:00:08
   -- ------------------------------------- 1.0/20.1 MB 4.1 MB/s eta 0:00:05
   ---- ----------------------------------- 2.1/20.1 MB 6.9 MB/s eta 0:00:03
   ------- -------------------------------- 3.5/20.1 MB 10.3 MB/s eta 0:00:02
   ---------- ----------------------------- 5.3/20.1 MB 13.5 MB/s eta 0:00:02
   ------------- --------

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Lenovo\\Desktop\\KYC_Project\\myenv\\Lib\\site-packages\\cv2\\cv2.pyd'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached onnxruntime-1.29.0-cp311-cp311-win_amd64.whl.metadata (5.8 kB)
Using cached onnxruntime-1.29.0-cp311-cp311-win_amd64.whl (14.0 MB)


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Lenovo\\Desktop\\KYC_Project\\myenv\\Lib\\site-packages\\onnxruntime\\capi\\onnxruntime_providers_shared.dll'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os

# Clone the repo directly
os.system("git clone https://github.com/minivision-ai/Silent-Face-Anti-Spoofing.git silent_face")

print("Done. Contents:")
print(os.listdir("silent_face"))

Done. Contents:
['.git']


In [5]:
import urllib.request
import os

# We only need the model weights, not the full repo
os.makedirs("silent_face/resources/anti_spoof_models", exist_ok=True)

models = {
    "2.7_80x80_MiniFASNetV2.pth": "https://github.com/minivision-ai/Silent-Face-Anti-Spoofing/raw/master/resources/anti_spoof_models/2.7_80x80_MiniFASNetV2.pth",
    "4_0_0_80x80_MiniFASNetV1SE.pth": "https://github.com/minivision-ai/Silent-Face-Anti-Spoofing/raw/master/resources/anti_spoof_models/4_0_0_80x80_MiniFASNetV1SE.pth",
}

for fname, url in models.items():
    save_path = f"silent_face/resources/anti_spoof_models/{fname}"
    print(f"Downloading {fname}...")
    try:
        urllib.request.urlretrieve(url, save_path)
        size = os.path.getsize(save_path)
        print(f"  Saved: {size/1024:.1f} KB")
    except Exception as e:
        print(f"  Failed: {e}")

print("\nDone:")
print(os.listdir("silent_face/resources/anti_spoof_models"))

  Saved: 1806.1 KB
  Saved: 1812.6 KB

Done:
['2.7_80x80_MiniFASNetV2.pth', '4_0_0_80x80_MiniFASNetV1SE.pth']


In [6]:
import urllib.request
import os

base_url = "https://raw.githubusercontent.com/minivision-ai/Silent-Face-Anti-Spoofing/master"

files = {
    "silent_face/src/__init__.py": f"{base_url}/src/__init__.py",
    "silent_face/src/model_lib/__init__.py": f"{base_url}/src/model_lib/__init__.py",
    "silent_face/src/model_lib/MiniFASNet.py": f"{base_url}/src/model_lib/MiniFASNet.py",
    "silent_face/src/utility.py": f"{base_url}/src/utility.py",
}

for local_path, url in files.items():
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"OK: {local_path}")
    except Exception as e:
        print(f"FAIL: {local_path} — {e}")

print("\nDone")

FAIL: silent_face/src/__init__.py — HTTP Error 404: Not Found
FAIL: silent_face/src/model_lib/__init__.py — HTTP Error 404: Not Found
OK: silent_face/src/model_lib/MiniFASNet.py
OK: silent_face/src/utility.py

Done


In [7]:
# Create empty __init__.py files
open("silent_face/src/__init__.py", "w").close()
open("silent_face/src/model_lib/__init__.py", "w").close()

print("Created __init__.py files")
print("\nFull structure:")
for root, dirs, files in os.walk("silent_face"):
    # skip .git
    if ".git" in root:
        continue
    level = root.replace("silent_face", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

Created __init__.py files

Full structure:
silent_face/
  resources/
    anti_spoof_models/
      2.7_80x80_MiniFASNetV2.pth
      4_0_0_80x80_MiniFASNetV1SE.pth
  src/
    utility.py
    __init__.py
    model_lib/
      MiniFASNet.py
      __init__.py


In [9]:
def load_model(model_name, device):
    config = MODEL_CONFIGS[model_name]
    model = config["model_class"](conv6_kernel=(5,5))
    model_path = os.path.join(MODEL_DIR, model_name)
    state = torch.load(model_path, map_location=device)

    if "state_dict" in state:
        state = state["state_dict"]

    # Remove module. prefix
    state = {k.replace("module.", ""): v for k, v in state.items()}

    # Fix SE module key mismatch
    new_state = {}
    for k, v in state.items():
        k = k.replace(".se_fc1.", ".se_module.fc1.")
        k = k.replace(".se_fc2.", ".se_module.fc2.")
        k = k.replace(".se_bn1.", ".se_module.bn1.")
        k = k.replace(".se_bn2.", ".se_module.bn2.")
        new_state[k] = v

    model.load_state_dict(new_state, strict=False)
    model.to(device)
    model.eval()
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model1 = load_model("2.7_80x80_MiniFASNetV2.pth", device)
model2 = load_model("4_0_0_80x80_MiniFASNetV1SE.pth", device)

print("Both liveness models loaded successfully")

Device: cpu
Both liveness models loaded successfully


In [10]:
import torchvision.transforms as transforms

# Preprocessing transform
transform = transforms.Compose([
    transforms.Resize((80, 80)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

def predict_liveness(model, face_img):
    """Run single model prediction on a face image."""
    if isinstance(face_img, np.ndarray):
        face_img = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
    
    tensor = transform(face_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        prob = torch.softmax(output, dim=1)
        # class 1 = real, class 0 = fake
        real_score = prob[0][1].item()
    
    return real_score

def check_liveness(selfie_path, threshold=0.6):
    """
    Full liveness check pipeline.
    Combines predictions from both models.
    """
    # Load image
    if isinstance(selfie_path, str):
        img = Image.open(selfie_path).convert("RGB")
    else:
        img = selfie_path

    # Get scores from both models
    score1 = predict_liveness(model1, img)
    score2 = predict_liveness(model2, img)

    # Average both scores
    final_score = (score1 + score2) / 2

    if final_score >= 0.6:
        label = "real"
    else:
        label = "spoof"

    return {
        "is_live": final_score >= threshold,
        "score": round(final_score, 4),
        "score_model1": round(score1, 4),
        "score_model2": round(score2, 4),
        "label": label
    }

print("Liveness functions ready")

Liveness functions ready


In [11]:
# Test with real face photo
result = check_liveness("test_face.jpg")

print("Liveness Detection Result:")
print(f"  Is Live:      {result['is_live']}")
print(f"  Final Score:  {result['score']}")
print(f"  Model 1:      {result['score_model1']}")
print(f"  Model 2:      {result['score_model2']}")
print(f"  Label:        {result['label']}")

Liveness Detection Result:
  Is Live:      False
  Final Score:  0.0166
  Model 1:      0.0055
  Model 2:      0.0277
  Label:        spoof


In [12]:
os.makedirs("ml/liveness-service/models", exist_ok=True)

# Copy model weights
import shutil
shutil.copy2(
    "silent_face/resources/anti_spoof_models/2.7_80x80_MiniFASNetV2.pth",
    "ml/liveness-service/models/2.7_80x80_MiniFASNetV2.pth"
)
shutil.copy2(
    "silent_face/resources/anti_spoof_models/4_0_0_80x80_MiniFASNetV1SE.pth",
    "ml/liveness-service/models/4_0_0_80x80_MiniFASNetV1SE.pth"
)

print("Models saved to ml/liveness-service/models/")
print(os.listdir("ml/liveness-service/models"))

Models saved to ml/liveness-service/models/
['2.7_80x80_MiniFASNetV2.pth', '4_0_0_80x80_MiniFASNetV1SE.pth']
